# Galaxy Flux Pipeline Example Notebook

This notebook demonstrates a reproducible, synthetic end-to-end test of key science components used by `galflux` with manuscript-style explanation.

Demo targets referenced throughout repository: **UGC 9024** and **NGC 6902**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from galaxy_flux_pipeline.masking import build_mask
from galaxy_flux_pipeline.psf_matching import match_psf
from galaxy_flux_pipeline.stacking import exposure_weighted_stack
from galaxy_flux_pipeline.validation import flux_stats
from galaxy_flux_pipeline.photometry import cps_to_mjy, nanomaggy_to_mjy


## 1) Build synthetic galaxy + stars scene
We create a smooth galaxy component and add compact contaminants to evaluate masking and PSF behavior.


In [ ]:
ny,nx=160,160
y,x=np.indices((ny,nx))
gal=np.exp(-(((x-80)/18)**2+((y-80)/12)**2))
stars=np.zeros_like(gal)
stars[30,30]=4
stars[120,110]=3
img=gal+stars+0.03*np.random.default_rng(3).normal(size=gal.shape)
fig,ax=plt.subplots(figsize=(5,4),dpi=140)
im=ax.imshow(img,origin='lower',cmap='magma')
ax.set_title('Synthetic science image')
plt.colorbar(im,ax=ax,label='arb. units'); plt.show()


## 2) Multi-stage mask with galaxy protection


In [ ]:
mask,seg=build_mask(img,nsigma=2.0,npixels=8,protect_center_px=24)
fig,axs=plt.subplots(1,3,figsize=(13,4),dpi=140)
axs[0].imshow(img,origin='lower',cmap='magma'); axs[0].set_title('Image')
axs[1].imshow(seg,origin='lower',cmap='viridis'); axs[1].set_title('Segmentation')
axs[2].imshow(img,origin='lower',cmap='gray')
axs[2].contour(mask,levels=[0.5],colors='cyan',linewidths=0.7)
axs[2].set_title('Mask overlay')
for a in axs: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()


## 3) PSF matching and flux conservation check


In [ ]:
pre=np.nan_to_num(img,copy=True)
post=match_psf(pre,current_fwhm=1.4,target_fwhm=5.3,pixscale=1.0)
stats=flux_stats(np.array([pre.sum()]),np.array([post.sum()]))
print('Integrated fractional flux stats:',stats)
fig,axs=plt.subplots(1,2,figsize=(10,4),dpi=140)
axs[0].imshow(pre,origin='lower',cmap='magma'); axs[0].set_title('Pre-PSF match')
axs[1].imshow(post,origin='lower',cmap='magma'); axs[1].set_title('Post-PSF match')
for a in axs: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()


## 4) Exposure-aware stacking demonstration


In [ ]:
im1=pre
im2=np.roll(pre,1,axis=0)
exp1=np.ones_like(pre)*100
exp2=np.ones_like(pre)*150
stack,exp,err=exposure_weighted_stack([im1,im2],[exp1,exp2])
print('Stack shape:',stack.shape,'Median err:',np.nanmedian(err))


## 5) Unit conversions


In [ ]:
print('GALEX FUV 1 cps -> mJy:', cps_to_mjy(np.array([1.0]),'FUV')[0])
print('1 nanomaggy -> mJy:', nanomaggy_to_mjy(np.array([1.0]))[0])


## Interpretation
This notebook is designed as a deterministic synthetic validation of major algorithmic pieces. For full science analysis, use real mission products via CLI workflows and inspect outputs under `outputs/<galaxy>/`.
